# Transforms and Enrichment with Progress Tracking

This notebook demonstrates:
- Adding computed transforms (CVD state, event inference)
- Progress bars that look great in Jupyter
- Caching enriched data for fast reloading
- Combining incremental + transforms

In [ ]:
from vulnflow import VulnDataset
import tempfile
from pathlib import Path

## Load Base Dataset

Start with raw NVD data:

In [ ]:
# Load sample NVD data
ds = VulnDataset.from_nvd("../tests/fixtures/nvd_sample.json")

print(f"Loaded {len(ds)} vulnerabilities")
print(f"Base columns: {len(ds.columns)}")

## Add CVD State Transform

CVDStateTransform derives 19 analytics fields tracking the coordinated vulnerability disclosure timeline:

In [ ]:
from vulnflow.transforms.computed.cvd_state import CVDStateTransform

# Apply CVD state transform
ds_cvd = ds.transform([CVDStateTransform()])

print(f"After CVD transform: {len(ds_cvd.columns)} columns")
print(f"New columns added: {len(ds_cvd.columns) - len(ds.columns)}")

## Explore CVD State Columns

CVD state includes dates, booleans, and computed gaps:

In [ ]:
df = ds_cvd.to_pandas()

# Show CVD state columns
cvd_cols = [col for col in df.columns if col.startswith('cvd_')]
print(f"CVD State Columns ({len(cvd_cols)} total):")
for col in sorted(cvd_cols)[:10]:  # Show first 10
    print(f"  - {col}")
print(f"  ... and {len(cvd_cols) - 10} more")

# Show first CVE's state
first_cve = df.iloc[0]
print(f"\n{first_cve['cve_id']} CVD State:")
print(f"  Vendor Aware (V): {first_cve['cvd_vendor_aware']}")
print(f"  Fix Released (F): {first_cve['cvd_fix_released']}")
print(f"  Public Disclosure (D): {first_cve['cvd_public_disclosure']}")
print(f"  Exploit Public (X): {first_cve['cvd_exploit_public']}")
print(f"  Attacks Observed (A): {first_cve['cvd_attacks_observed']}")

## Add Event Inference

EventInference uses heuristics to infer CVD events from references and timestamps:

In [ ]:
from vulnflow.transforms.inferences.events import EventInference

# Apply event inference
ds_events = ds.transform([CVDStateTransform(), EventInference()])

df_events = ds_events.to_pandas()

# Show event columns
event_cols = [col for col in df_events.columns if col.startswith('event_')]
print(f"Event Inference Columns ({len(event_cols)} total):")
for col in sorted(event_cols):
    print(f"  - {col}")

## Progress Bars in Jupyter

VulnFlow shows clean progress bars when running transforms:

In [ ]:
# Run multiple transforms to see progress
from vulnflow.transforms.inferences.tags import TagInference

transforms = [
    CVDStateTransform(),
    EventInference(),
    TagInference(),
]

print("Running 3 transforms...\n")
ds_multi = VulnDataset.from_nvd("../tests/fixtures/nvd_sample.json").transform(transforms)
print(f"\nComplete! {len(ds_multi.columns)} total columns")

## Cache Enriched Data

Save transformed data to avoid recomputing every time:

In [ ]:
# Save enriched dataset
demo_dir = Path(tempfile.mkdtemp())
enriched_path = demo_dir / "nvd_enriched.parquet"

ds_multi.to_parquet(enriched_path)
print(f"Saved enriched data: {enriched_path}")

In [ ]:
# Fast reload (no recomputation needed)
import time

start = time.time()
df_cached = VulnDataset.from_parquet(enriched_path).to_pandas()
elapsed = time.time() - start

print(f"Loaded enriched data in {elapsed:.3f}s")
print(f"  Records: {len(df_cached)}")
print(f"  Columns: {len(df_cached.columns)}")
print(f"\nAll transforms preserved!")

## Complete Workflow Example

Putting it all together:

In [ ]:
# Step 1: Load raw NVD data
raw_ds = VulnDataset.from_nvd("../tests/fixtures/nvd_sample.json")
print(f"1. Loaded {len(raw_ds)} CVEs with {len(raw_ds.columns)} base columns")

# Step 2: Apply transforms
enriched_ds = raw_ds.transform([
    CVDStateTransform(),
    EventInference(),
    TagInference(),
])
print(f"2. Applied 3 transforms → {len(enriched_ds.columns)} columns")

# Step 3: Cache for fast reloading
cache_path = demo_dir / "workflow_cache.parquet"
enriched_ds.to_parquet(cache_path)
print(f"3. Cached to: {cache_path}")

# Step 4: Next session - instant load
df_final = VulnDataset.from_parquet(cache_path).to_pandas()
print(f"4. Reloaded {len(df_final)} records with all enrichments")

# Step 5: Analyze!
print(f"\nReady for analysis with {len(df_final.columns)} columns")

## Incremental + Transforms

Combine incremental loading with transforms:

In [ ]:
# Build cache with transforms (one-time)
cache_transformed = demo_dir / "cache_transformed.parquet"
VulnDataset.from_nvd("../tests/fixtures/nvd_sample.json") \
    .transform([CVDStateTransform(), EventInference()]) \
    .to_parquet(cache_transformed)

print(f"Built transformed cache: {cache_transformed}")

# Every session: merge cache + delta (both pre-transformed)
# In production, also transform the delta before saving
df_incremental = VulnDataset.from_parquet(cache_transformed).to_pandas()
print(f"Loaded {len(df_incremental)} records with transforms")

In [ ]:
# Cleanup
import shutil
shutil.rmtree(demo_dir)
print(f"Cleaned up: {demo_dir}")

## Production Pattern

```python
# Build transformed cache (one-time)
from vulnflow import VulnDataset
from vulnflow.transforms.computed import CVDStateTransform
from vulnflow.transforms.inferences import EventInference, TagInference

transforms = [CVDStateTransform(), EventInference(), TagInference()]

VulnDataset.from_nvd("nvdcve-2.0-Recent.json") \
    .transform(transforms) \
    .to_parquet("nvd_cache_transformed.parquet")

# Process delta with same transforms
VulnDataset.from_nvd("nvdcve-2.0-Modified.json") \
    .transform(transforms) \
    .to_parquet("nvd_delta_transformed.parquet")

# Every session: instant load with all enrichments
df = VulnDataset.from_parquet([
    "nvd_cache_transformed.parquet",
    "nvd_delta_transformed.parquet"
]).to_pandas()
```

## Key Takeaways

✓ **CVD State** - 19 analytics fields tracking disclosure timeline  
✓ **Event Inference** - Infer CVD events from references and timestamps  
✓ **Progress tracking** - Clean progress bars in Jupyter  
✓ **Cache enrichments** - Save transformed data to parquet  
✓ **Combine patterns** - Incremental + transforms for fast, rich datasets  

## Available Transforms

**Computed (sync):**
- `CVDStateTransform` - 19 CVD analytics fields

**Inference (sync with confidence):**
- `EventInference` - Infer CVD events from timestamps/references  
- `TagInference` - Infer exploit/patch tags from references

**Enrichers (async with external data):**
- `KEVEnricher` - CISA Known Exploited Vulnerabilities (requires network)
- `EPSSEnricher` - Exploit Prediction Scoring System (requires network)